In [1]:
!pip -q install azure-ai-documentintelligence azure-core
!pip install azure-storage-blob

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.0/106.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.8/217.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 20.6 MB/s eta 0:00:00


In [2]:
"""
Batch OCR of Amazon-style invoices in an Azure Blob container
using Azure AI Document Intelligence (Prebuilt Read)
"""

from openpyxl import Workbook
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest
from azure.storage.blob import ContainerClient
import numpy as np
import re
import socket

print("running")
socket.gethostbyname("canadatest.cognitiveservices.azure.com")

# Secrets (Kaggle)
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

endpoint = user_secrets.get_secret("AZURE_DOCINT_ENDPOINT")
key = user_secrets.get_secret("AZURE_DOCINT_KEY")
container_sas_url = user_secrets.get_secret("AZURE_BLOB_CONTAINER_SAS_URL")

print("Endpoint loaded:", bool(endpoint))
print("Key loaded:", bool(key))
print("Container SAS loaded:", bool(container_sas_url))

# Clients
docint_client = DocumentIntelligenceClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key)
)

container_client = ContainerClient.from_container_url(container_sas_url)

# Helpers
def extract_invoice_fields(text: str) -> dict:
    """
    Extract key Amazon invoice fields from OCR text
    """
    fields = {
        "order_number": None,
        "order_date": None,
        "order_total": None,
        "grand_total": None,
        "line_items": []
    }

    # Order number
    m = re.search(r"Order\s+#?(\d{3}-\d{7}-\d{7})", text)
    if m:
        fields["order_number"] = m.group(1)

    # Order date
    m = re.search(r"Order Placed:\s*([A-Za-z]+\s+\d{1,2},\s+\d{4})", text)
    if m:
        fields["order_date"] = m.group(1)

    # Order total
    m = re.search(r"Order Total:\s*\$([\d\.]+)", text)
    if m:
        fields["order_total"] = float(m.group(1))

    # Grand total
    m = re.search(r"Grand Total:\s*\$([\d\.]+)", text)
    if m:
        fields["grand_total"] = float(m.group(1))

    # Line items (simple heuristic)
    for line in text.splitlines():
        price_match = re.search(r"\$([\d]+\.\d{2})$", line.strip())
        if price_match and "Sold by" not in line:
            fields["line_items"].append({
                "description": line.strip(),
                "price": float(price_match.group(1))
            })

    return fields

# Main OCR Loop
def analyze_all_documents(sheet):
    base_url, sas = container_sas_url.split("?", 1)

    for blob in container_client.list_blobs():
        if not blob.name.lower().endswith((".pdf", ".png", ".jpg", ".jpeg")):
            continue

        blob_url = f"{base_url}/{blob.name}?{sas}"
        print("\n" + "=" * 60)
        print(f"📄 Processing Document: {blob.name}")
        print("=" * 60)

        poller = docint_client.begin_analyze_document(
            "prebuilt-read",
            AnalyzeDocumentRequest(url_source=blob_url)
        )

        result = poller.result()

        full_text = result.content or ""
        fields = extract_invoice_fields(full_text)

        # Structured Output
        print("Order Number :", fields["order_number"])
        print("Order Date   :", fields["order_date"])
        print("Order Total  :", fields["order_total"])
        print("Grand Total  :", fields["grand_total"])

        print("\nLine Items:")
        for item in fields["line_items"]:
            print(f" - ${item['price']:.2f} | {item['description'][:80]}")


def create_excel_workbook():
    workbook = Workbook()
    sheet = workbook.active
    sheet.title = "Amazon Invoices"

    sheet.append([
        "source_file",
        "order_number",
        "order_date",
        "order_total",
        "grand_total",
        "item_description",
        "item_price"
    ])

    return workbook, sheet

def write_invoice_to_excel(sheet, source_file, fields):
    for item in fields["line_items"]:
        sheet.append([
            source_file,
            fields["order_number"],
            fields["order_date"],
            fields["order_total"],
            fields["grand_total"],
            item["description"],
            item["price"]
        ])

def save_excel(workbook, path):
    workbook.save(path)

# Entry Point
if __name__ == "__main__":
    workbook, sheet = create_excel_workbook()

    analyze_all_documents(sheet)

    output_path = "amazon_invoice_ocr_results.xlsx"
    workbook.save(output_path)

    print(f"\n📁 Excel file written: {output_path}")

running
Endpoint loaded: True
Key loaded: True
Container SAS loaded: True

📄 Processing Document: Amazon/Amazon01.pdf
Order Number : 113-8456210-7723491
Order Date   : November 12, 2025
Order Total  : 27.84
Grand Total  : 27.84

Line Items:
 - $27.84 | Details for Order #113-8456210-7723491 Order Placed: November 12, 2025 Order Tot
 - $15.98 | $15.98
 - $11.86 | $11.86
 - $27.84 | $27.84
 - $0.00 | $0.00
 - $0.00 | $0.00
 - $27.84 | $27.84

📄 Processing Document: Amazon/Amazon02.pdf
Order Number : 114-2098841-5510927
Order Date   : December 3, 2025
Order Total  : 29.73
Grand Total  : 29.73

Line Items:
 - $29.73 | Details for Order #114-2098841-5510927 Order Placed: December 3, 2025 Order Tota
 - $9.49 | $9.49
 - $12.99 | $12.99
 - $7.25 | $7.25
 - $29.73 | $29.73
 - $0.00 | $0.00
 - $0.00 | $0.00
 - $29.73 | $29.73

📄 Processing Document: Amazon/Amazon03.pdf
Order Number : 112-7734501-0049182
Order Date   : January 7, 2026
Order Total  : 21.29
Grand Total  : 21.29

Line Items:
 - $21.